# Task 4 Game Title Detection: local Mac and Kaggle

This notebook is prepared for both local Jupyter and Kaggle. It validates dataset paths, image references, class mapping, and preprocessing before training. Set `RUN_TRAINING = True` only after the setup cells succeed.

Local Mac dependency command:

```bash
python -m pip install torch torchvision pillow pandas scikit-learn
```

Kaggle normally provides PyTorch and torchvision. Source images in this project are 230x120 RGB JPEGs; the model transform resizes them to 224x224 for ResNet18.

In [11]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 342
RUN_TRAINING = True
RUNTIME_READY = False
try:
    import torch
    from PIL import Image
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from torchvision import models, transforms
    RUNTIME_READY = True
except ImportError as error:
    print('Training dependencies are unavailable; setup validation can still run.')
    print('Install torch, torchvision, pillow, pandas, and scikit-learn before setting RUN_TRAINING=True.')

SEARCH_ROOTS = [Path('/kaggle/input'), Path.cwd(), Path.cwd().parent]
candidates = []
for root in SEARCH_ROOTS:
    if root.exists():
        candidates.extend([p for p in root.rglob('train.csv')
                           if (p.parent / 'test.csv').is_file()
                           and (p.parent / 'train').is_dir()
                           and (p.parent / 'test').is_dir()])
# Deduplicate paths and prefer the official image-classification directory when several inputs are mounted.
candidates = sorted(set(p.parent for p in candidates), key=lambda p: (('04_image_classification' not in str(p)), str(p)))
if not candidates:
    raise FileNotFoundError('No Task 4 directory found. Expected train.csv, test.csv, train/, and test/.')
TASK_DIR = candidates[0]
TRAIN_DIR, TEST_DIR = TASK_DIR / 'train', TASK_DIR / 'test'
train_csv = pd.read_csv(TASK_DIR / 'train.csv')
test_csv = pd.read_csv(TASK_DIR / 'test.csv')
if not {'id', 'class'}.issubset(train_csv.columns) or 'id' not in test_csv.columns:
    raise ValueError(f'Unexpected columns: train={train_csv.columns.tolist()}, test={test_csv.columns.tolist()}')
if train_csv.id.isna().any() or train_csv.id.duplicated().any() or test_csv.id.isna().any() or test_csv.id.duplicated().any():
    raise ValueError('Train and test IDs must be non-null and unique')
classes = sorted(train_csv['class'].astype(str).unique().tolist())
class_to_idx = {name: idx for idx, name in enumerate(classes)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}
image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
train_paths = {p.stem: p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in image_extensions}
test_paths = {p.stem: p for p in TEST_DIR.iterdir() if p.suffix.lower() in image_extensions}
missing_train = sorted(set(train_csv.id.astype(str)) - set(train_paths))
missing_test = sorted(set(test_csv.id.astype(str)) - set(test_paths))
if missing_train or missing_test:
    raise FileNotFoundError(f'Missing image references: train={len(missing_train)}, test={len(missing_test)}')
print('Task directory:', TASK_DIR)
print('Train/test shapes:', train_csv.shape, test_csv.shape)
print('Class mapping:', class_to_idx)


Task directory: /Users/dinoden/PlayerIntelSystem/Player_Intelligence_System/cpe342-kaggle-main/04_image_classification
Train/test shapes: (31546, 2) (25889, 2)
Class mapping: {'7-11k': 0, 'fiveN': 1, 'freefried': 2, 'pube': 3, 'roblock': 4}


In [12]:
if RUNTIME_READY:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class GameImages(Dataset):
        def __init__(self, frame, paths, transform, labelled=True):
            self.frame = frame.reset_index(drop=True)
            self.paths = paths
            self.transform = transform
            self.labelled = labelled
        def __len__(self): return len(self.frame)
        def __getitem__(self, index):
            row = self.frame.iloc[index]
            with Image.open(self.paths[str(row.id)]) as image:
                image = image.convert('RGB')
            tensor = self.transform(image)
            if self.labelled: return tensor, class_to_idx[str(row['class'])]
            return tensor, str(row.id)

    train_indices, valid_indices = train_test_split(np.arange(len(train_csv)), test_size=.20, random_state=SEED, stratify=train_csv['class'])
    train_transform = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    eval_transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    train_ds = GameImages(train_csv.iloc[train_indices], train_paths, train_transform)
    valid_ds = GameImages(train_csv.iloc[valid_indices], train_paths, eval_transform)
    test_ds = GameImages(test_csv, test_paths, eval_transform, labelled=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0 if DEVICE.type == 'cpu' else 2)
    valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=False, num_workers=0 if DEVICE.type == 'cpu' else 2)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=0 if DEVICE.type == 'cpu' else 2)
    print('Device:', DEVICE, 'split sizes:', len(train_ds), len(valid_ds), len(test_ds))
else:
    print('Dataset and schema validation completed; image loaders are deferred until dependencies are installed.')


Device: cpu split sizes: 25236 6310 25889


In [13]:
if RUNTIME_READY and RUN_TRAINING:
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    except Exception as error:
        print('Pretrained weights unavailable; using random initialization:', error)
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    best_f1 = -1.0
    for epoch in range(5):
        model.train()
        for images, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images.to(DEVICE)), labels.to(DEVICE))
            loss.backward(); optimizer.step()
        model.eval(); true, pred = [], []
        with torch.no_grad():
            for images, labels in valid_loader:
                pred.extend(model(images.to(DEVICE)).argmax(1).cpu().numpy()); true.extend(labels.numpy())
        valid_f1 = f1_score(true, pred, average='macro')
        print(f'epoch={epoch+1} valid_macro_f1={valid_f1:.4f}')
        if valid_f1 > best_f1:
            best_f1 = valid_f1
            torch.save({'model_state_dict': model.state_dict(), 'class_to_idx': class_to_idx, 'best_macro_f1': best_f1}, Path.cwd() / 'task4_best_model.pt')
else:
    print('Training is disabled. Set RUN_TRAINING=True only after validating the setup and dependencies.')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/dinoden/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


epoch=1 valid_macro_f1=0.9074
epoch=2 valid_macro_f1=0.9178
epoch=3 valid_macro_f1=0.9167
epoch=4 valid_macro_f1=0.9392
epoch=5 valid_macro_f1=0.9301


In [14]:
if RUNTIME_READY and RUN_TRAINING:
    checkpoint = torch.load(Path.cwd() / 'task4_best_model.pt', map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict']); model.eval()
    ids, predictions = [], []
    with torch.no_grad():
        for images, batch_ids in test_loader:
            predictions.extend(model(images.to(DEVICE)).argmax(1).cpu().tolist()); ids.extend(list(batch_ids))
    output = pd.DataFrame({'id': ids, 'task4': [idx_to_class[int(i)] for i in predictions]})
    if output.id.tolist() != test_csv.id.astype(str).tolist(): raise ValueError('Output IDs do not preserve test.csv order')
    if not set(output.task4).issubset(set(classes)): raise ValueError('Invalid output labels')
    output.to_csv(Path.cwd() / 'task4_test_predictions.csv', index=False)
    Path.cwd().joinpath('task4_class_to_idx.json').write_text(json.dumps(class_to_idx, indent=2))
    print('Wrote task4_test_predictions.csv and task4_class_to_idx.json')
else:
    print('Inference is disabled because training has not been run.')


Wrote task4_test_predictions.csv and task4_class_to_idx.json


In [15]:
from pathlib import Path

print("Working directory:", Path.cwd())

for name in [
    "task4_test_predictions.csv",
    "task4_class_to_idx.json",
    "task4_best_model.pt",
]:
    path = Path.cwd() / name
    print(name, "exists:", path.exists(), "path:", path)

Working directory: /Users/dinoden/PlayerIntelSystem/Player_Intelligence_System/notebook
task4_test_predictions.csv exists: True path: /Users/dinoden/PlayerIntelSystem/Player_Intelligence_System/notebook/task4_test_predictions.csv
task4_class_to_idx.json exists: True path: /Users/dinoden/PlayerIntelSystem/Player_Intelligence_System/notebook/task4_class_to_idx.json
task4_best_model.pt exists: True path: /Users/dinoden/PlayerIntelSystem/Player_Intelligence_System/notebook/task4_best_model.pt


## Output handoff

On Kaggle, download `task4_test_predictions.csv` and copy it to the project `predictions/` directory. Copy `task4_class_to_idx.json` and `task4_best_model.pt` to `artifacts/` if retaining the model. On local Mac, the same outputs are written to the notebook working directory. This notebook has not started training; `RUN_TRAINING` defaults to `False`.